Importing Libraries

In [161]:
import pandas as pd
import numpy as np
import pickle

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, classification_report

read data

In [162]:
users=pd.read_csv("data/users.csv")
hotels = pd.read_csv("data/hotels.csv")

print(f"Shape of Users Dataset: {users.shape}")
print(f"Shape of Hotel Dataset: {hotels.shape}")

Shape of Users Dataset: (1340, 5)
Shape of Hotel Dataset: (40552, 8)


In [163]:
#Before fix
print("Gender distribution BEFORE fix:")
print(users['gender'].value_counts())

users['gender'] = users['gender'].replace('none', 'unknown')

#After fix
print("Gender distribution AFTER fix:")
print(users['gender'].value_counts())

Gender distribution BEFORE fix:
gender
male      452
female    448
none      440
Name: count, dtype: int64
Gender distribution AFTER fix:
gender
male       452
female     448
unknown    440
Name: count, dtype: int64


Merge Users and Hotels Data

In [164]:
#rename name in Hotel to hotel_name
hotels.rename(columns={"name":"hotel_name"},inplace=True)

#Merging the Datasets
df=hotels.merge(users[["code","age","gender",'company']],left_on='userCode',right_on='code',how='left')

#droping Useless columns
df.drop(columns=['userCode','code','travelCode','total'],inplace=True)

print(f"Merged dataset shape: {df.shape}")

Merged dataset shape: (40552, 8)


Feature Engineering

In [165]:
#creating Month Column
df['date']=pd.to_datetime(df['date'],errors='coerce')
df['month']=df['date'].dt.month

#drop date column
df.drop(columns=['date'],inplace=True)


In [166]:
def price_tier(price):
    if price<=180: return "budget"
    elif price<=250: return "mid"
    else: return "luxury"
    
df['price_tier']=df['price'].apply(price_tier)

Price and Place are Removed this leads to Leakage of Data

In [167]:
# 'price' → unique per hotel → perfectly predicts hotel name (leakage)
# 'place' → unique per hotel → same problem

df.drop(columns=['place','price'],inplace=True)


In [168]:
print("Before Removing of Null Values",df.shape)
df.dropna(inplace=True)
print("After Removing of Null Values",df.shape)
print("Remaining columns:", df.columns.tolist())

Before Removing of Null Values (40552, 7)
After Removing of Null Values (40552, 7)
Remaining columns: ['hotel_name', 'days', 'age', 'gender', 'company', 'month', 'price_tier']


Encoding Categorical Columns

In [169]:
# One-Hot Encode company → one binary column per company
company_dummies = pd.get_dummies(df['company'], prefix='co').astype(int)
df = pd.concat([df.drop('company', axis=1), company_dummies], axis=1)
company_cols = list(company_dummies.columns)

print('Company columns:', company_cols)
print()

# Label encode gender and price_tier
encoders = {}
for col in ['gender', 'price_tier']:
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col])
    encoders[col] = le

# Encode target
hotel_encoder = LabelEncoder()
df['hotel_name'] = hotel_encoder.fit_transform(df['hotel_name'])

print('Gender classes :', list(encoders['gender'].classes_))
print('Price tier     :', list(encoders['price_tier'].classes_))
print('Hotels         :', list(hotel_encoder.classes_))

Company columns: ['co_4You', 'co_Acme Factory', 'co_Monsters CYA', 'co_Umbrella LTDA', 'co_Wonka Company']



Gender classes : ['female', 'male', 'unknown']
Price tier     : ['budget', 'luxury', 'mid']
Hotels         : ['Hotel A', 'Hotel AF', 'Hotel AU', 'Hotel BD', 'Hotel BP', 'Hotel BW', 'Hotel CB', 'Hotel K', 'Hotel Z']


Training Test Split

In [170]:
numeric_features     = ['days', 'month', 'age']
categorical_features = ['gender', 'price_tier'] + company_cols
target               = 'hotel_name'

In [171]:
X = df[numeric_features + categorical_features]
y = df[target]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Training samples : {len(X_train)}")
print(f"Testing  samples : {len(X_test)}")

Training samples : 32441
Testing  samples : 8111


Training Model

In [172]:
model = RandomForestClassifier(
    n_estimators   = 300,   # more trees = more stable and accurate predictions
    max_depth      = 20,    # deeper to capture company-hotel patterns
    min_samples_leaf = 2,   # prevents tiny noisy leaf nodes
    random_state   = 42,
    n_jobs         = -1     # use all CPU cores
)

model.fit(X_train, y_train)
print('Model training complete')

Model training complete


In [173]:
predictions   = model.predict(X_test)
probabilities = model.predict_proba(X_test)
y_true        = y_test.values

def topk_accuracy(proba, y_true, k):
    """What fraction of test samples have the correct hotel in the top-k predictions?"""
    top_k = np.argsort(proba, axis=1)[:, -k:]  # top-k hotel indices per row
    hits  = sum(y_true[i] in top_k[i] for i in range(len(y_true)))
    return hits / len(y_true)


print('      RECOMMENDATION SYSTEM METRICS')

print(f'  Random baseline (9 hotels)  :  11.11%')
print(f'  Top-1 Accuracy (exact match): {topk_accuracy(probabilities, y_true, 1):.2%}  (3.3x better than random)')
print(f'  Top-3 Accuracy              : {topk_accuracy(probabilities, y_true, 3):.2%}  -> Always correct!')
print(f'  Top-5 Accuracy              : {topk_accuracy(probabilities, y_true, 5):.2%}  -> Always correct!')
print('=' * 55)

      RECOMMENDATION SYSTEM METRICS
  Random baseline (9 hotels)  :  11.11%
  Top-1 Accuracy (exact match): 37.00%  (3.3x better than random)
  Top-3 Accuracy              : 100.00%  -> Always correct!
  Top-5 Accuracy              : 100.00%  -> Always correct!


Evaluation of the model

In [174]:
predictions = model.predict(X_test)

acc = accuracy_score(y_test, predictions)
print(f"Overall Accuracy: {acc:.2%}")
print()

# Detailed per-hotel breakdown
print("Per-hotel Performance:")
print(classification_report(
    y_test, predictions,
    target_names=hotel_encoder.classes_
))

Overall Accuracy: 37.00%

Per-hotel Performance:
              precision    recall  f1-score   support

     Hotel A       0.37      0.38      0.37       666
    Hotel AF       0.35      0.37      0.36       966
    Hotel AU       0.38      0.35      0.36       893
    Hotel BD       0.38      0.41      0.40       966
    Hotel BP       0.37      0.37      0.37       887
    Hotel BW       0.34      0.34      0.34       867
    Hotel CB       0.36      0.35      0.36      1006
     Hotel K       0.42      0.46      0.44      1019
     Hotel Z       0.31      0.29      0.30       841

    accuracy                           0.37      8111
   macro avg       0.37      0.37      0.37      8111
weighted avg       0.37      0.37      0.37      8111



In [175]:
feature_names = numeric_features + categorical_features
importances   = model.feature_importances_

importance_df = pd.DataFrame({
    'Feature'   : feature_names,
    'Importance': importances
}).sort_values('Importance', ascending=False)

print("Feature Importance (higher = more influential):")
print(importance_df.to_string(index=False))

Feature Importance (higher = more influential):
         Feature  Importance
      price_tier    0.415142
             age    0.308453
           month    0.158989
            days    0.054338
          gender    0.036224
 co_Monsters CYA    0.007361
 co_Acme Factory    0.005234
co_Umbrella LTDA    0.005043
         co_4You    0.004882
co_Wonka Company    0.004334


Saving the Model

In [176]:
artifacts = {
    'model'               : model,
    'hotel_encoder'       : hotel_encoder,
    'encoders'            : encoders,           # gender, price_tier encoders
    'company_cols'        : company_cols,        # one-hot column names for company
    'numeric_features'    : numeric_features,
    'categorical_features': categorical_features
}

with open('feature_recommender.pkl', 'wb') as f:
    pickle.dump(artifacts, f)

print('Model saved: feature_recommender.pkl')

Model saved: feature_recommender.pkl


In [177]:
ALL_COMPANIES = ['4You', 'Acme Factory', 'Monsters CYA', 'Umbrella LTDA', 'Wonka Company']

def recommend_hotel(days, month, age, gender, budget, company, top_n=5):
    """
    Recommend top hotels for a new user.

    Parameters
    ----------
    days    : int — Days staying (e.g., 3)
    month   : int — Travel month 1-12
    age     : int — User age
    gender  : str — 'male', 'female', or 'unknown'
    budget  : str — 'budget', 'mid', or 'luxury'
    company : str — Company name (must match training data)
    top_n   : int — How many hotels to recommend
    """
    with open('feature_recommender.pkl', 'rb') as f:
        arts = pickle.load(f)

    mdl       = arts['model']
    h_enc     = arts['hotel_encoder']
    encs      = arts['encoders']
    co_cols   = arts['company_cols']
    num_feats = arts['numeric_features']
    cat_feats = arts['categorical_features']

    # Validate inputs
    if gender not in list(encs['gender'].classes_):
        print(f'Invalid gender. Options: {list(encs["gender"].classes_)}')
        return []
    if budget not in list(encs['price_tier'].classes_):
        print(f'Invalid budget. Options: {list(encs["price_tier"].classes_)}')
        return []
    if company not in ALL_COMPANIES:
        print(f'Unknown company. Options: {ALL_COMPANIES}')
        return []

    # Encode inputs
    enc_gender = encs['gender'].transform([gender])[0]
    enc_budget = encs['price_tier'].transform([budget])[0]

    # One-hot encode company
    company_values = {col: int(col == f'co_{company}') for col in co_cols}

    # Build input DataFrame
    row = {'days': days, 'month': month, 'age': age,
           'gender': enc_gender, 'price_tier': enc_budget}
    row.update(company_values)
    input_df = pd.DataFrame([row], columns=num_feats + cat_feats)

    # Predict probabilities
    proba = mdl.predict_proba(input_df)[0]

    # Rank top-N
    top_indices = proba.argsort()[-top_n:][::-1]
    return [
        {'rank': i + 1,
         'hotel': h_enc.inverse_transform([idx])[0],
         'match_score': round(float(proba[idx]) * 100, 2)}
        for i, idx in enumerate(top_indices)
    ]

In [ ]:
def print_recs(recs, label):
    
    for r in recs:
        print(f"  Rank {r['rank']}: {r['hotel']:<12}")
    print()

# Test 1: Acme Factory, luxury — should NOT recommend Hotel A (company policy)
recs = recommend_hotel(days=4, month=6, age=28, gender='male',
                       budget='luxury', company='Acme Factory', top_n=5)
print_recs(recs, 'Young male | Acme Factory | Luxury')

# Test 2: 4You, budget
recs = recommend_hotel(days=7, month=12, age=55, gender='female',
                       budget='budget', company='4You', top_n=5)
print_recs(recs, 'Senior female | 4You | Budget')

# Test 3: Unknown gender
recs = recommend_hotel(days=3, month=3, age=35, gender='unknown',
                       budget='mid', company='Monsters CYA', top_n=5)
print_recs(recs, 'Unknown gender | Monsters CYA | Mid')

  Rank 1: Hotel AU    
  Rank 2: Hotel K     
  Rank 3: Hotel BP    
  Rank 4: Hotel Z     
  Rank 5: Hotel CB    

  Rank 1: Hotel AF    
  Rank 2: Hotel CB    
  Rank 3: Hotel BW    
  Rank 4: Hotel BD    
  Rank 5: Hotel AU    

  Rank 1: Hotel BP    
  Rank 2: Hotel Z     
  Rank 3: Hotel BD    
  Rank 4: Hotel A     
  Rank 5: Hotel K     



: 